# 00 — EHRSHOT trusted foundation

This notebook reruns the validated aggregate foundation on the real downloaded release. It never displays patient rows, identifiers, timestamps, or histories.

## Governance and pinned configuration

In [ ]:
import json
import platform
from importlib.metadata import version
from pathlib import Path

from IPython.display import display

from temporal_ehr.config import load_config, logical_path, resolve_path
from temporal_ehr.data.audit import classify_readiness, run_data_audit
from temporal_ehr.data.censoring import validate_strict_censoring, write_censoring_reports
from temporal_ehr.data.cohort import summarize_existing_cohort
from temporal_ehr.data.inventory import inventory_assets
from temporal_ehr.data.label_validation import validate_labels, write_label_reports
from temporal_ehr.data.reporting import write_reports
from temporal_ehr.data.task_registry import load_task_registry

data_path = resolve_path(None, "EHRSHOT_DATA_PATH", must_be_file=True)
asset_root = resolve_path(None, "EHRSHOT_ASSET_ROOT", must_be_dir=True)
cohort_path = Path("outputs/private/cohorts/primary_panel.parquet")
if not cohort_path.is_file():
    command = (
        ".venv/bin/python -m temporal_ehr.cli build-cohort "
        '--asset-root "${EHRSHOT_ASSET_ROOT}" --tasks lab_anemia new_acutemi guo_los '
        "--selection-seed 20260814 --private-output outputs/private/cohorts/primary_panel.parquet "
        "--allow-private-output"
    )
    print(f"Missing governed cohort. Rebuild it exactly with:\n{command}")
    raise FileNotFoundError("primary_panel.parquet is required; no replacement cohort was created")
expected = load_config()
registry = load_task_registry(Path("configs/tasks.yaml"))
primary_tasks = tuple(registry["primary_panel"])
pinned = {
    "logical_event_file": logical_path(data_path, asset_root),
    "logical_asset_root": asset_root.name,
    "logical_cohort_file": cohort_path.name,
    "primary_tasks": primary_tasks,
    "modeling_splits": ["train", "validation"],
    "strict_censoring": "event_time < prediction_time",
    "expected_event_sha256": expected["dataset"]["sha256"],
    "official_repository_commit": expected["official_repository"]["commit"],
}
display(pinned)

## Environment and package versions

In [ ]:
packages = ["temporal-ehr-ml4h", "duckdb", "pyarrow", "numpy", "scikit-learn", "scipy", "nbclient", "nbformat"]
display(
    {
        "python_version": platform.python_version(),
        "operating_system": platform.system(),
        "machine_architecture": platform.machine(),
        "python_packages": {name: version(name) for name in packages},
    }
)

## Existing aggregate audit and validation

In [ ]:
audit_dir = Path("reports/generated/data_audit")
inventory = inventory_assets(asset_root, data_path, expected)
audit = run_data_audit(data_path, asset_root, audit_dir, expected)
readiness = classify_readiness(audit, inventory)
audit["readiness"] = readiness
inventory["readiness"] = readiness
write_reports(audit, inventory, audit_dir)
label_report = validate_labels(asset_root, registry)
write_label_reports(label_report, Path("reports/generated/label_validation"))
cohort_report = summarize_existing_cohort(cohort_path, primary_tasks)
censoring_report = validate_strict_censoring(data_path, cohort_path)
write_censoring_reports(censoring_report, Path("reports/generated/censoring"))
display(
    {
        "event_audit_flags": readiness["flags"],
        "label_validation": label_report["status"],
        "cohort": cohort_report["status"],
        "strict_censoring": censoring_report["status"],
    }
)

## Primary-panel aggregate counts

In [ ]:
primary_counts = [row for row in cohort_report["groups"] if row["task"] in primary_tasks]
display(primary_counts)

## Source-row versus patient-balanced prevalence

Source-row prevalence weights every qualifying label row. The governed cohort instead contributes one row per patient and task, so frequent testers do not receive more weight merely because they were measured more often.

In [ ]:
prevalence_comparison = []
for task in primary_tasks:
    source_splits = label_report["tasks"][task]["split_summary"].values()
    source_rows = sum(group["rows"] for group in source_splits)
    source_positive = sum(group["positive_rows"] for group in source_splits)
    cohort_groups = [group for group in cohort_report["groups"] if group["task"] == task]
    cohort_rows = sum(group["rows"] for group in cohort_groups)
    cohort_positive = sum(group["positive_rows"] for group in cohort_groups)
    prevalence_comparison.append(
        {
            "task": task,
            "all_source_label_rows": source_rows,
            "source_row_prevalence": source_positive / source_rows,
            "one_row_per_patient_rows": cohort_rows,
            "patient_balanced_prevalence": cohort_positive / cohort_rows,
            "prevalence_difference_patient_balanced_minus_source": cohort_positive / cohort_rows
            - source_positive / source_rows,
        }
    )
display(prevalence_comparison)

## Strict-censoring exclusions

In [ ]:
exact_time_exclusions = [
    {
        "task": row["task"],
        "split": row["split"],
        "instances": row["instances"],
        "instances_with_exact_time_events": row["instances_with_exact_time_events"],
        "exact_time_event_rows_excluded": row["exact_time_event_rows_excluded"],
    }
    for row in censoring_report["aggregate_groups"]
    if row["task"] in primary_tasks
]
display(exact_time_exclusions)

## Machine-readable readiness checklist

In [ ]:
available_groups = {(row["task"], row["split"]) for row in cohort_report["groups"] if row["rows"] >= 20}
checklist = {
    "event_table_valid": readiness["event_table_valid"],
    "label_validation_passed": label_report["status"] == "PASS",
    "governed_primary_cohort_present": cohort_path.is_file(),
    "primary_tasks_exact": list(primary_tasks) == ["lab_anemia", "new_acutemi", "guo_los"],
    "train_validation_groups_at_least_20": all(
        (task, split) in available_groups for task in primary_tasks for split in ("train", "validation")
    ),
    "strict_censoring_passed": censoring_report["status"] == "PASS",
    "test_allowed_for_modeling": False,
}
required_checks = [
    "event_table_valid",
    "label_validation_passed",
    "governed_primary_cohort_present",
    "primary_tasks_exact",
    "train_validation_groups_at_least_20",
    "strict_censoring_passed",
]
checklist["ready_for_count_baseline"] = all(checklist[name] for name in required_checks)
checklist["ready_for_serialization_qc_except_local_qwen_tokenizer"] = checklist["ready_for_count_baseline"]
print(json.dumps(checklist, sort_keys=True))